# Bio-Rad Gate Model: Training and Prediction

Run the cells in order with the `learn_python` kernel. This notebook reproduces the training and prediction scripts with a grouped holdout and 400-tree Random Forest.

Reference labels are ClusterData-mean midpoints or amplitude-bound estimates, not original saved manual gate coordinates. Estimated labels are included in training and evaluation. All-well predictions include training wells and must not be used as independent test results. Saved model and CSV artifacts are overwritten.

## 1. Import Dependencies and Define Paths

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline

PROJECT_DIR = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / 'Biorad_manual_gating').is_dir()
     and (candidate / 'data_output').is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Open this notebook from within the project directory.')
OUTPUT_DIR = PROJECT_DIR / 'model_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUTPUT_DIR / 'biorad_gate_model.joblib'
METRICS_PATH = OUTPUT_DIR / 'biorad_gate_model_metrics.json'
TEST_PREDICTIONS_PATH = OUTPUT_DIR / 'biorad_gate_model_predictions.csv'
IMPORTANCE_PATH = OUTPUT_DIR / 'biorad_gate_feature_importance.csv'
PREDICTION_OUTPUT_PATH = OUTPUT_DIR / 'biorad_gate_predictions_all_wells.csv'
TEST_SIZE = 0.2
RANDOM_STATE = 42
print(f'Project: {PROJECT_DIR}\nOutput: {OUTPUT_DIR}')

Project: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating
Output: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\model_output


In [2]:
"""2. Constants and reusable helpers."""
TARGET_COLUMNS = ['x_gate', 'y_gate']
GROUP_COLUMN = 'parent_dataset'
IDENTIFIER_COLUMNS = ['dataset', 'parent_dataset', 'well']
LEAKED_COLUMNS = {
    'x_gate', 'y_gate', 'target1(MUT)', 'target2(WT)', 'quadrant',
    'count', 'ch1_mean', 'ch2_mean', 'gate_has_x', 'gate_has_y',
    'gate_n_cluster_rows', 'gate_total_cluster_count',
    'gate_ch1_mean_weighted', 'gate_ch2_mean_weighted',
    'n_populated_quadrants', 'weighted_mean_ch1_MUT', 'weighted_mean_ch2_WT',
}


def default_data_path() -> Path:
    return PROJECT_DIR / 'data_output' / 'full_biorad_cluster_dataset.csv'


def select_feature_columns(data: pd.DataFrame) -> list[str]:
    excluded = LEAKED_COLUMNS | set(IDENTIFIER_COLUMNS)
    return [column for column in data.columns
            if column not in excluded and pd.api.types.is_numeric_dtype(data[column])]


DATA_PATH = default_data_path()
INFERENCE_DATA_PATH = DATA_PATH
print(f'Training input: {DATA_PATH}\nInference input: {INFERENCE_DATA_PATH}')

Training input: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\data_output\full_biorad_cluster_dataset.csv
Inference input: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\data_output\full_biorad_cluster_dataset.csv


In [3]:
"""3. Load and validate training data."""
data = pd.read_csv(DATA_PATH)
required_columns = TARGET_COLUMNS + IDENTIFIER_COLUMNS
missing_columns = [column for column in required_columns if column not in data]
if missing_columns:
    raise ValueError(f'Required columns are missing: {missing_columns}')
input_rows = len(data)
data[TARGET_COLUMNS] = data[TARGET_COLUMNS].apply(pd.to_numeric, errors='coerce')
data = data.dropna(subset=TARGET_COLUMNS + [GROUP_COLUMN]).copy()
print(f'Input wells: {input_rows}; eligible wells: {len(data)}; dropped: {input_rows - len(data)}')
method_columns = [column for column in ['x_gate_method', 'y_gate_method'] if column in data]
for column in method_columns:
    display(data[column].value_counts(dropna=False).rename_axis(column).to_frame('wells'))
display(data.head())

Input wells: 8713; eligible wells: 8713; dropped: 0


,wells
x_gate_method,
cluster_means,8532
single_observed_class_amplitude_bound,181


,wells
y_gate_method,
cluster_means,6351
single_observed_class_amplitude_bound,2362


,dataset,parent_dataset,well,x_gate,y_gate,x_gate_method,y_gate_method,x_min_ch2,x_max_ch2,y_min_ch1,...,ch2_iqr,ch2_cv,ch2_skew,ch2_kurt,ch2_bimodality,ch2_log_std,ch2_log_skew,ch2_outlier_frac,ch2_kde_peaks,ch1_ch2_corr
0,Rapeseed TRT-13_TRT001_A01_Amplitude,Rapeseed TRT-13_TRT001,A01,1525.437550,3438.880518,cluster_means,cluster_means,362.2588,3112.239,300.7066,...,1281.36400,0.384056,-0.022360,-1.844808,0.866090,0.410681,-0.098693,0.0,2,0.395736
1,Rapeseed TRT-13_TRT001_A02_Amplitude,Rapeseed TRT-13_TRT001,A02,1658.720934,2724.028515,cluster_means,cluster_means,834.7704,3166.599,1662.3340,...,1283.39850,0.338357,-0.420272,-1.586671,0.832523,0.387740,-0.523402,0.0,2,0.373964
2,Rapeseed TRT-13_TRT001_A03_Amplitude,Rapeseed TRT-13_TRT001,A03,1686.486856,4119.919418,cluster_means,cluster_means,844.3401,3003.538,1668.9320,...,1246.63150,0.329829,-0.493250,-1.522239,0.841337,0.381758,-0.594342,0.0,2,0.337172
3,Rapeseed TRT-13_TRT001_A04_Amplitude,Rapeseed TRT-13_TRT001,A04,1669.179179,2809.077206,cluster_means,cluster_means,847.4105,3176.465,1648.6370,...,335.33350,0.246442,-1.133239,-0.008224,0.763503,0.305406,-1.375745,0.0,2,0.256095
4,Rapeseed TRT-13_TRT001_A05_Amplitude,Rapeseed TRT-13_TRT001,A05,1755.753028,2688.398641,cluster_means,cluster_means,864.9615,3054.688,1698.9040,...,1248.86925,0.359014,-0.116416,-1.788716,0.836759,0.388903,-0.206399,0.0,2,0.317222


In [4]:
"""4. Select leakage-resistant numeric features."""
feature_columns = select_feature_columns(data)
if not feature_columns:
    raise ValueError('No numeric, non-leaking feature columns were found.')
assert not LEAKED_COLUMNS.intersection(feature_columns)
print(f'Selected {len(feature_columns)} features:')
print(feature_columns)

Selected 28 features:
['x_min_ch2', 'x_max_ch2', 'y_min_ch1', 'y_max_ch1', 'amplitude_range_WT', 'amplitude_range_MUT', 'n_droplets', 'ch1_std', 'ch1_iqr', 'ch1_cv', 'ch1_skew', 'ch1_kurt', 'ch1_bimodality', 'ch1_log_std', 'ch1_log_skew', 'ch1_outlier_frac', 'ch1_kde_peaks', 'ch2_std', 'ch2_iqr', 'ch2_cv', 'ch2_skew', 'ch2_kurt', 'ch2_bimodality', 'ch2_log_std', 'ch2_log_skew', 'ch2_outlier_frac', 'ch2_kde_peaks', 'ch1_ch2_corr']


In [5]:
"""5. Group-aware train/test split."""
groups = data[GROUP_COLUMN].astype(str)
if groups.nunique() < 2:
    raise ValueError('At least two parent_dataset groups are required for evaluation.')
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(data[feature_columns], data[TARGET_COLUMNS], groups))
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
display(pd.DataFrame({
    'split': ['train', 'test'],
    'rows': [len(train_idx), len(test_idx)],
    'groups': [groups.iloc[train_idx].nunique(), groups.iloc[test_idx].nunique()],
}))

,split,rows,groups
0,train,6895,73
1,test,1818,19


In [6]:
"""6. Train the multi-output Random Forest pipeline."""
model = make_pipeline(
    SimpleImputer(strategy='median'),
    RandomForestRegressor(
        n_estimators=400, min_samples_leaf=3, max_features=0.8,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
)
model.fit(data.iloc[train_idx][feature_columns], data.iloc[train_idx][TARGET_COLUMNS])
test_predictions = model.predict(data.iloc[test_idx][feature_columns])
print(f'Trained on {len(train_idx)} wells; predicted {len(test_idx)} held-out wells.')

Trained on 6895 wells; predicted 1818 held-out wells.


In [7]:
"""7. Evaluate against the training-median baseline and build diagnostic tables."""
test_targets = data.iloc[test_idx][TARGET_COLUMNS].to_numpy()
baseline = np.tile(data.iloc[train_idx][TARGET_COLUMNS].median().to_numpy(), (len(test_idx), 1))
metrics = {
    'data_path': str(DATA_PATH), 'feature_columns': feature_columns,
    'target_columns': TARGET_COLUMNS, 'group_column': GROUP_COLUMN,
    'train_rows': int(len(train_idx)), 'test_rows': int(len(test_idx)),
    'train_groups': int(groups.iloc[train_idx].nunique()),
    'test_groups': int(groups.iloc[test_idx].nunique()),
}
for index, target in enumerate(TARGET_COLUMNS):
    metrics[f'model_mae_{target}'] = float(mean_absolute_error(test_targets[:, index], test_predictions[:, index]))
    metrics[f'model_rmse_{target}'] = float(mean_squared_error(test_targets[:, index], test_predictions[:, index]) ** 0.5)
    metrics[f'baseline_mae_{target}'] = float(mean_absolute_error(test_targets[:, index], baseline[:, index]))

predictions_df = data.iloc[test_idx][IDENTIFIER_COLUMNS + TARGET_COLUMNS].copy()
predictions_df['predicted_x_gate'] = test_predictions[:, 0]
predictions_df['predicted_y_gate'] = test_predictions[:, 1]
predictions_df['x_gate_error'] = predictions_df['predicted_x_gate'] - predictions_df['x_gate']
predictions_df['y_gate_error'] = predictions_df['predicted_y_gate'] - predictions_df['y_gate']
forest = model.named_steps['randomforestregressor']
importance_df = pd.DataFrame({'feature': feature_columns, 'importance': forest.feature_importances_}).sort_values('importance', ascending=False)
display(pd.DataFrame({
    'target': TARGET_COLUMNS,
    'model_mae': [metrics[f'model_mae_{target}'] for target in TARGET_COLUMNS],
    'model_rmse': [metrics[f'model_rmse_{target}'] for target in TARGET_COLUMNS],
    'baseline_mae': [metrics[f'baseline_mae_{target}'] for target in TARGET_COLUMNS],
}))
display(importance_df.head(15))
display(predictions_df.head())

,target,model_mae,model_rmse,baseline_mae
0,x_gate,356.699396,695.742373,803.271967
1,y_gate,334.417300,504.656451,1458.046902


,feature,importance
3,y_max_ch1,0.406627
2,y_min_ch1,0.393437
1,x_max_ch2,0.070711
7,ch1_std,0.024613
5,amplitude_range_MUT,0.014545
4,amplitude_range_WT,0.010962
0,x_min_ch2,0.009840
9,ch1_cv,0.009659
8,ch1_iqr,0.008959
13,ch1_log_std,0.008193


,dataset,parent_dataset,well,x_gate,y_gate,predicted_x_gate,predicted_y_gate,x_gate_error,y_gate_error
288,210121_TRT13_TRT008_(F1-R1)_A01_Amplitude,210121_TRT13_TRT008_(F1-R1),A01,1774.882744,2489.974395,1876.717552,2210.359851,101.834808,-279.614544
289,210121_TRT13_TRT008_(F1-R1)_A02_Amplitude,210121_TRT13_TRT008_(F1-R1),A02,1915.414721,1998.315109,2260.221093,2244.920632,344.806372,246.605523
290,210121_TRT13_TRT008_(F1-R1)_A03_Amplitude,210121_TRT13_TRT008_(F1-R1),A03,1971.694566,2012.608398,2011.474706,2331.337017,39.780140,318.728619
291,210121_TRT13_TRT008_(F1-R1)_A04_Amplitude,210121_TRT13_TRT008_(F1-R1),A04,1616.151372,2057.297660,2016.929779,2118.612121,400.778407,61.314461
292,210121_TRT13_TRT008_(F1-R1)_A05_Amplitude,210121_TRT13_TRT008_(F1-R1),A05,2152.262607,1999.763841,1969.930285,2117.538536,-182.332322,117.774695


In [8]:
"""8. Save the model and evaluation artifacts."""
joblib.dump(model, MODEL_PATH)
with METRICS_PATH.open('w', encoding='utf-8') as metrics_file:
    json.dump(metrics, metrics_file, indent=2)
predictions_df.to_csv(TEST_PREDICTIONS_PATH, index=False)
importance_df.to_csv(IMPORTANCE_PATH, index=False)
for path in [MODEL_PATH, METRICS_PATH, TEST_PREDICTIONS_PATH, IMPORTANCE_PATH]:
    print(f'Saved: {path}')
for target in TARGET_COLUMNS:
    print(f"{target} MAE: {metrics[f'model_mae_{target}']:.3f}; baseline: {metrics[f'baseline_mae_{target}']:.3f}")

Saved: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\model_output\biorad_gate_model.joblib
Saved: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\model_output\biorad_gate_model_metrics.json
Saved: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\model_output\biorad_gate_model_predictions.csv
Saved: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\model_output\biorad_gate_feature_importance.csv
x_gate MAE: 356.699; baseline: 803.272
y_gate MAE: 334.417; baseline: 1458.047


In [9]:
"""9. Load inference data and validate required features."""
inference_data = pd.read_csv(INFERENCE_DATA_PATH)
with METRICS_PATH.open('r', encoding='utf-8') as metrics_file:
    saved_metrics = json.load(metrics_file)
inference_features = saved_metrics['feature_columns']
missing_columns = [column for column in inference_features if column not in inference_data]
if missing_columns:
    raise ValueError(f'Required feature columns are missing: {missing_columns}')
print(f'Inference wells: {len(inference_data)}; required features: {len(inference_features)}')

Inference wells: 8713; required features: 28


In [10]:
"""10. Run batch prediction with the saved model and export the results."""
saved_model = joblib.load(MODEL_PATH)
batch_predictions = saved_model.predict(inference_data[inference_features])
result = inference_data.copy()
result['predicted_x_gate'] = batch_predictions[:, 0]
result['predicted_y_gate'] = batch_predictions[:, 1]
assert np.isfinite(batch_predictions).all()
assert len(result) == len(inference_data)
result.to_csv(PREDICTION_OUTPUT_PATH, index=False)
print(f'Saved predictions: {PREDICTION_OUTPUT_PATH}\nPredicted wells: {len(result)}')
preview_columns = [column for column in IDENTIFIER_COLUMNS if column in result] + ['predicted_x_gate', 'predicted_y_gate']
display(result[preview_columns].head(10))

Saved predictions: C:\Users\HQCHEJUN\Downloads\github_projects\ddPCR_ML_Biorad_manual_gating\model_output\biorad_gate_predictions_all_wells.csv
Predicted wells: 8713


,dataset,parent_dataset,well,predicted_x_gate,predicted_y_gate
0,Rapeseed TRT-13_TRT001_A01_Amplitude,Rapeseed TRT-13_TRT001,A01,1749.542183,3334.058153
1,Rapeseed TRT-13_TRT001_A02_Amplitude,Rapeseed TRT-13_TRT001,A02,1651.363479,2768.152399
2,Rapeseed TRT-13_TRT001_A03_Amplitude,Rapeseed TRT-13_TRT001,A03,1725.684726,3697.016958
3,Rapeseed TRT-13_TRT001_A04_Amplitude,Rapeseed TRT-13_TRT001,A04,1725.911876,3054.919228
4,Rapeseed TRT-13_TRT001_A05_Amplitude,Rapeseed TRT-13_TRT001,A05,1690.676528,2679.514761
5,Rapeseed TRT-13_TRT001_A06_Amplitude,Rapeseed TRT-13_TRT001,A06,1708.214389,2677.270143
6,Rapeseed TRT-13_TRT001_A07_Amplitude,Rapeseed TRT-13_TRT001,A07,1584.801489,2992.996390
7,Rapeseed TRT-13_TRT001_A08_Amplitude,Rapeseed TRT-13_TRT001,A08,1629.289857,2674.460006
8,Rapeseed TRT-13_TRT001_A09_Amplitude,Rapeseed TRT-13_TRT001,A09,1597.805817,2878.991060
9,Rapeseed TRT-13_TRT001_A10_Amplitude,Rapeseed TRT-13_TRT001,A10,1683.255163,3447.863610


## 11. Held-Out Droplet Classification and MUT/WT Impact

This evaluation reloads saved held-out predictions and raw amplitude CSVs; it does not retrain the model. Run the import/path configuration cell before these cells.

A target is positive when its amplitude is strictly greater than its gate (equality is negative). Quadrants are Q1=(1,1), Q2=(1,0), Q3=(0,0), Q4=(0,1), ordered Target 1=MUT then Target 2=WT. Reference assignments use reconstructed dataset gates, not original Bio-Rad per-droplet labels. Amplitude-bound reference gates can make agreement look optimistic.

Positive-fraction shifts are reported in percentage points. Poisson occupancy is -log(1-positive fraction), in copies per droplet, not copies per microlitre. All-positive wells have unbounded occupancy and are left blank in occupancy comparisons. Neither absolute concentration nor a biological MUT/WT decision can be validated without droplet volume, dilution, controls, and an agreed acceptance threshold.

In [11]:
def assign_quadrants(amplitude: np.ndarray, x_gate: float, y_gate: float) -> np.ndarray:
    mut_positive = amplitude[:, 0] > y_gate
    wt_positive = amplitude[:, 1] > x_gate
    return np.where(mut_positive, np.where(wt_positive, 1, 2), np.where(wt_positive, 4, 3))


def compare_droplets(amplitude: np.ndarray, reference_x: float, reference_y: float,
                     predicted_x: float, predicted_y: float) -> tuple[dict, np.ndarray]:
    reference = assign_quadrants(amplitude, reference_x, reference_y)
    predicted = assign_quadrants(amplitude, predicted_x, predicted_y)
    confusion = np.zeros((4, 4), dtype=np.int64)
    np.add.at(confusion, (reference - 1, predicted - 1), 1)
    summary = {
        'n_droplets': len(amplitude),
        'changed_droplets': int(np.sum(reference != predicted)),
        'quadrant_agreement_pct': float(100 * np.mean(reference == predicted)),
    }
    for quadrant in range(1, 5):
        reference_count = int(np.sum(reference == quadrant))
        predicted_count = int(np.sum(predicted == quadrant))
        summary[f'reference_Q{quadrant}_count'] = reference_count
        summary[f'predicted_Q{quadrant}_count'] = predicted_count
        summary[f'Q{quadrant}_count_change'] = predicted_count - reference_count
    for target, quadrants in [('MUT', [1, 2]), ('WT', [1, 4])]:
        reference_positive = np.isin(reference, quadrants)
        predicted_positive = np.isin(predicted, quadrants)
        reference_fraction = float(reference_positive.mean())
        predicted_fraction = float(predicted_positive.mean())
        summary[f'reference_{target}_positive_count'] = int(reference_positive.sum())
        summary[f'predicted_{target}_positive_count'] = int(predicted_positive.sum())
        summary[f'reference_{target}_positive_fraction'] = reference_fraction
        summary[f'predicted_{target}_positive_fraction'] = predicted_fraction
        summary[f'{target}_positive_shift_pp'] = 100 * (predicted_fraction - reference_fraction)
        summary[f'{target}_changed_droplets'] = int(np.sum(reference_positive != predicted_positive))
        reference_occupancy = float(-np.log1p(-reference_fraction)) if reference_fraction < 1 else np.nan
        predicted_occupancy = float(-np.log1p(-predicted_fraction)) if predicted_fraction < 1 else np.nan
        summary[f'reference_{target}_copies_per_droplet'] = reference_occupancy
        summary[f'predicted_{target}_copies_per_droplet'] = predicted_occupancy
        summary[f'{target}_copies_per_droplet_change'] = predicted_occupancy - reference_occupancy
    return summary, confusion


synthetic_amplitude = np.array([[20., 20.], [20., 0.], [0., 0.], [0., 20.], [10., 10.]])
assert assign_quadrants(synthetic_amplitude, 10., 10.).tolist() == [1, 2, 3, 4, 3]
same, same_confusion = compare_droplets(synthetic_amplitude, 10., 10., 10., 10.)
assert same['changed_droplets'] == 0 and same['quadrant_agreement_pct'] == 100
changed, changed_confusion = compare_droplets(synthetic_amplitude, 10., 10., 30., 10.)
assert changed['changed_droplets'] == 2
assert changed['MUT_positive_shift_pp'] == 0 and np.isclose(changed['WT_positive_shift_pp'], -40)
assert changed_confusion[0, 1] == 1 and changed_confusion[3, 2] == 1
print('Synthetic quadrant, equality, unchanged-gate, and WT-shift checks passed.')

Synthetic quadrant, equality, unchanged-gate, and WT-shift checks passed.


In [12]:
assert PROJECT_DIR is not None
heldout = pd.read_csv(TEST_PREDICTIONS_PATH)
reference_data = pd.read_csv(DATA_PATH)
label_columns = IDENTIFIER_COLUMNS + ['x_gate_method', 'y_gate_method']
heldout = heldout.merge(reference_data[label_columns], on=IDENTIFIER_COLUMNS, how='left', validate='one_to_one')
amplitude_index = {}
for amplitude_path in (PROJECT_DIR / 'ddPCR_data').rglob('*_Amplitude.csv'):
    key = (amplitude_path.parent.name, amplitude_path.stem)
    amplitude_index.setdefault(key, []).append(amplitude_path)

well_rows = []
failed_rows = []
pooled_confusion = np.zeros((4, 4), dtype=np.int64)
for position, row in enumerate(heldout.to_dict('records'), start=1):
    identity = {column: row[column] for column in IDENTIFIER_COLUMNS}
    candidates = amplitude_index.get((str(row['parent_dataset']), str(row['dataset'])), [])
    if len(candidates) != 1:
        failed_rows.append({**identity, 'reason': f'Expected one amplitude CSV, found {len(candidates)}'})
        continue
    try:
        gates = np.array([row[column] for column in ['x_gate', 'y_gate', 'predicted_x_gate', 'predicted_y_gate']], dtype=float)
        if not np.isfinite(gates).all():
            raise ValueError('Nonfinite gate coordinates')
        frame = pd.read_csv(candidates[0], skiprows=3, low_memory=False)
        amplitude = np.asarray(frame[['Ch1Amplitude', 'Ch2Amplitude']].apply(pd.to_numeric, errors='coerce'), dtype=float)
        valid = np.isfinite(amplitude).all(axis=1)
        excluded_points = int((~valid).sum())
        amplitude = amplitude[valid]
        if not len(amplitude):
            raise ValueError('No finite amplitude observations')
        summary, confusion = compare_droplets(amplitude, *gates)
    except (OSError, ValueError, KeyError, pd.errors.ParserError) as error:
        failed_rows.append({**identity, 'reason': str(error)})
        continue
    well_rows.append({
        **identity, 'x_gate_method': row['x_gate_method'], 'y_gate_method': row['y_gate_method'],
        'x_gate': gates[0], 'y_gate': gates[1], 'predicted_x_gate': gates[2], 'predicted_y_gate': gates[3],
        'excluded_amplitude_rows': excluded_points, **summary,
    })
    pooled_confusion += confusion
    if position % 250 == 0:
        print(f'Evaluated {position}/{len(heldout)} held-out wells')

well_comparison = pd.DataFrame(well_rows)
if well_comparison.empty:
    raise ValueError('No held-out wells could be evaluated; inspect failed_rows.')
well_comparison['changed_fraction'] = well_comparison['changed_droplets'] / well_comparison['n_droplets']
well_comparison['reference_gate_type'] = np.where(
    (well_comparison[['x_gate_method', 'y_gate_method']] == 'cluster_means').all(axis=1),
    'both_cluster_means', 'includes_estimate_or_unknown',
)
experiment_rows = []
for experiment, wells in well_comparison.groupby('parent_dataset'):
    droplets = int(wells['n_droplets'].sum())
    summary = {
        'parent_dataset': experiment, 'evaluated_wells': len(wells), 'n_droplets': droplets,
        'changed_droplets': int(wells['changed_droplets'].sum()),
        'pooled_quadrant_agreement_pct': 100 * (1 - wells['changed_droplets'].sum() / droplets),
        'mean_well_quadrant_agreement_pct': wells['quadrant_agreement_pct'].mean(),
        'worst_well_quadrant_agreement_pct': wells['quadrant_agreement_pct'].min(),
    }
    for target in ['MUT', 'WT']:
        summary[f'mean_abs_{target}_positive_shift_pp'] = wells[f'{target}_positive_shift_pp'].abs().mean()
        summary[f'pooled_{target}_positive_shift_pp'] = 100 * (
            wells[f'predicted_{target}_positive_count'].sum() - wells[f'reference_{target}_positive_count'].sum()
        ) / droplets
        summary[f'mean_abs_{target}_copies_per_droplet_change'] = wells[f'{target}_copies_per_droplet_change'].abs().mean()
    experiment_rows.append(summary)
experiment_comparison = pd.DataFrame(experiment_rows).sort_values('pooled_quadrant_agreement_pct')
confusion_df = pd.DataFrame(pooled_confusion, index=['Q1', 'Q2', 'Q3', 'Q4'], columns=['Q1', 'Q2', 'Q3', 'Q4'])
confusion_df.index.name = 'reference_quadrant'
confusion_df.columns.name = 'predicted_quadrant'
well_comparison.to_csv(OUTPUT_DIR / 'heldout_droplet_comparison.csv', index=False)
experiment_comparison.to_csv(OUTPUT_DIR / 'heldout_experiment_comparison.csv', index=False)
confusion_df.to_csv(OUTPUT_DIR / 'heldout_quadrant_confusion.csv')
pd.DataFrame(failed_rows, columns=IDENTIFIER_COLUMNS + ['reason']).to_csv(OUTPUT_DIR / 'heldout_droplet_evaluation_failures.csv', index=False)
assert len(well_comparison) + len(failed_rows) == len(heldout)
assert pooled_confusion.sum() == well_comparison['n_droplets'].sum()
assert np.trace(pooled_confusion) == (well_comparison['n_droplets'] - well_comparison['changed_droplets']).sum()
print(f'Evaluated {len(well_comparison)}/{len(heldout)} wells; failures: {len(failed_rows)}')
print(f"Droplets evaluated: {well_comparison['n_droplets'].sum():,}")

Evaluated 250/1818 held-out wells


Evaluated 500/1818 held-out wells


Evaluated 750/1818 held-out wells


Evaluated 1000/1818 held-out wells


Evaluated 1250/1818 held-out wells


Evaluated 1500/1818 held-out wells


Evaluated 1750/1818 held-out wells


Evaluated 1818/1818 wells; failures: 0
Droplets evaluated: 27,755,560


In [13]:
total_droplets = int(well_comparison['n_droplets'].sum())
evaluation_summary = {
    'evaluated_wells': len(well_comparison), 'failed_wells': len(failed_rows),
    'evaluated_droplets': total_droplets,
    'pooled_quadrant_agreement_pct': float(100 * np.trace(pooled_confusion) / total_droplets),
    'median_well_quadrant_agreement_pct': float(well_comparison['quadrant_agreement_pct'].median()),
    'p10_well_quadrant_agreement_pct': float(well_comparison['quadrant_agreement_pct'].quantile(0.1)),
    'wells_at_least_99pct_agreement': int((well_comparison['quadrant_agreement_pct'] >= 99).sum()),
}
for target in ['MUT', 'WT']:
    shifts = well_comparison[f'{target}_positive_shift_pp'].abs()
    evaluation_summary[f'mean_abs_{target}_positive_shift_pp'] = float(shifts.mean())
    evaluation_summary[f'p95_abs_{target}_positive_shift_pp'] = float(shifts.quantile(0.95))
    evaluation_summary[f'max_abs_{target}_positive_shift_pp'] = float(shifts.max())
    evaluation_summary[f'{target}_wells_unbounded_occupancy'] = int(
        ((well_comparison[f'reference_{target}_positive_fraction'] == 1)
         | (well_comparison[f'predicted_{target}_positive_fraction'] == 1)).sum()
    )
with (OUTPUT_DIR / 'heldout_droplet_evaluation_summary.json').open('w', encoding='utf-8') as summary_file:
    json.dump(evaluation_summary, summary_file, indent=2)
print(json.dumps(evaluation_summary, indent=2))
display(confusion_df)
display(well_comparison.groupby('reference_gate_type')['quadrant_agreement_pct'].agg(['count', 'mean', 'median', 'min']))
display(experiment_comparison.head(5))
worst_columns = IDENTIFIER_COLUMNS + ['quadrant_agreement_pct', 'MUT_positive_shift_pp', 'WT_positive_shift_pp']
display(well_comparison.sort_values('quadrant_agreement_pct')[worst_columns].head(10))

{
  "evaluated_wells": 1818,
  "failed_wells": 0,
  "evaluated_droplets": 27755560,
  "pooled_quadrant_agreement_pct": 95.96146862106187,
  "median_well_quadrant_agreement_pct": 99.86087577947495,
  "p10_well_quadrant_agreement_pct": 91.92314540629395,
  "wells_at_least_99pct_agreement": 1290,
  "mean_abs_MUT_positive_shift_pp": 2.0005506111926987,
  "p95_abs_MUT_positive_shift_pp": 1.600049899563362,
  "max_abs_MUT_positive_shift_pp": 99.9568717653824,
  "MUT_wells_unbounded_occupancy": 0,
  "mean_abs_WT_positive_shift_pp": 2.2768368903721883,
  "p95_abs_WT_positive_shift_pp": 10.380657484411998,
  "max_abs_WT_positive_shift_pp": 100.0,
  "WT_wells_unbounded_occupancy": 77
}


predicted_quadrant,Q1,Q2,Q3,Q4
reference_quadrant,,,,
Q1,7270,465,390,14312
Q2,76,1776,1050,9
Q3,768,75186,10862148,193732
Q4,431475,154,403300,15763449


,count,mean,median,min
reference_gate_type,,,,
both_cluster_means,1500,97.787456,99.875034,2.257735
includes_estimate_or_unknown,318,86.013290,99.372070,0.000000


,parent_dataset,evaluated_wells,n_droplets,changed_droplets,pooled_quadrant_agreement_pct,mean_well_quadrant_agreement_pct,worst_well_quadrant_agreement_pct,mean_abs_MUT_positive_shift_pp,pooled_MUT_positive_shift_pp,mean_abs_MUT_copies_per_droplet_change,mean_abs_WT_positive_shift_pp,pooled_WT_positive_shift_pp,mean_abs_WT_copies_per_droplet_change
8,210802_MIP-043_TRT63_TA171,96,1369681,445408,67.480895,67.211174,0.043128,32.767140,31.977884,1.256463,0.077907,0.076806,0.004539
11,211001_TRT65_TA223,96,1572996,179807,88.569138,88.507025,43.296870,0.032174,-0.032931,0.000322,11.469078,-11.405687,0.241742
12,220104_MIP-064_TRT63_TA253,96,1421785,149987,89.450796,89.585997,0.000000,0.016338,-0.015403,0.000164,10.397723,10.532183,6.829748
0,191003_TRT2_TRT015,95,960856,46475,95.163167,94.884743,59.677232,0.185490,-0.216057,0.001867,4.945374,-4.544594,0.134260
14,220408_MIP-067_FF28_TA274,96,1427415,52202,96.342900,96.342996,83.936384,0.008173,-0.002452,0.000082,3.650839,-3.650655,0.075760


,dataset,parent_dataset,well,quadrant_agreement_pct,MUT_positive_shift_pp,WT_positive_shift_pp
1079,220104_MIP-064_TRT63_TA253_C03_Amplitude,220104_MIP-064_TRT63_TA253,C03,0.000000,0.000000,100.000000
1724,210802_MIP-043_TRT63_TA171_A03_Amplitude,210802_MIP-043_TRT63_TA171,A03,0.043128,99.956872,0.000000
1476,220531_MIP-082-MC_TRT76_TA341_D04_Amplitude,220531_MIP-082-MC_TRT76_TA341,D04,0.049451,0.000000,99.950549
1067,220104_MIP-064_TRT63_TA253_B03_Amplitude,220104_MIP-064_TRT63_TA253,B03,0.052677,0.000000,99.947323
1464,220531_MIP-082-MC_TRT76_TA341_C04_Amplitude,220531_MIP-082-MC_TRT76_TA341,C04,0.091565,0.000000,99.908435
1801,210802_MIP-043_TRT63_TA171_G08_Amplitude,210802_MIP-043_TRT63_TA171,G08,0.316498,99.676768,0.033670
1780,210802_MIP-043_TRT63_TA171_E11_Amplitude,210802_MIP-043_TRT63_TA171,E11,0.378100,99.621900,0.015124
1756,210802_MIP-043_TRT63_TA171_C11_Amplitude,210802_MIP-043_TRT63_TA171,C11,0.507469,99.492531,0.028590
1786,210802_MIP-043_TRT63_TA171_F05_Amplitude,210802_MIP-043_TRT63_TA171,F05,0.720158,99.279842,0.101909
1799,210802_MIP-043_TRT63_TA171_G06_Amplitude,210802_MIP-043_TRT63_TA171,G06,0.742421,99.257579,0.013749
